# WP2 T2.5 DBRepo Load and Verification

Owner C task: verify the T2.1 schema is in 3NF, prepare code to load the input data into DBRepo through the REST API, and verify that all T2.4 views return correct results.

In [65]:
from pathlib import Path
import os
import requests
import pandas as pd
import numpy as np
from dbrepo.RestClient import RestClient

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

DBREPO_BASE_URL = os.getenv("DBREPO_BASE_URL", "https://test.dbrepo.tuwien.ac.at").rstrip("/")
DBREPO_USER = os.getenv("DBREPO_USER")
DBREPO_PASSWORD = os.getenv("DBREPO_PASSWORD")
DBREPO_DB_ID = os.getenv("DBREPO_DB_ID", "3d81c073-e5fd-49b9-9536-b75ed490ca3e")
API_BASE = f"{DBREPO_BASE_URL}/api/v1"
AUTH = (DBREPO_USER, DBREPO_PASSWORD)
CLIENT = RestClient(endpoint=DBREPO_BASE_URL, username=DBREPO_USER, password=DBREPO_PASSWORD)

if not DBREPO_USER or not DBREPO_PASSWORD:
    raise RuntimeError("Set DBREPO_USER and DBREPO_PASSWORD as environment variables or in a local .env file.")

SOURCE_FILES = {
    "collision": {
        "file": "dft-road-casualty-statistics-collision-2023.csv",
        "url": "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-2023.csv",
    },
    "vehicle": {
        "file": "dft-road-casualty-statistics-vehicle-2023.csv",
        "url": "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-vehicle-2023.csv",
    },
    "casualty": {
        "file": "dft-road-casualty-statistics-casualty-2023.csv",
        "url": "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-casualty-2023.csv",
    },
}

LOAD_TO_DBREPO = os.getenv("LOAD_TO_DBREPO", "false").lower() == "true"
ALLOW_NONEMPTY_LOAD = os.getenv("ALLOW_NONEMPTY_LOAD", "false").lower() == "true"
MAX_ROWS_PER_TABLE = os.getenv("MAX_ROWS_PER_TABLE")
MAX_ROWS_PER_TABLE = int(MAX_ROWS_PER_TABLE) if MAX_ROWS_PER_TABLE else None

print(f"DBRepo API: {API_BASE}")
print(f"Database: {DBREPO_DB_ID}")
print(f"LOAD_TO_DBREPO={LOAD_TO_DBREPO}")

DBRepo API: https://test.dbrepo.tuwien.ac.at/api/v1
Database: 3d81c073-e5fd-49b9-9536-b75ed490ca3e
LOAD_TO_DBREPO=False


## Download or Reuse Official STATS19 CSV Files

In [66]:
def download_if_missing(name, spec):
    path = DATA_RAW / spec["file"]
    if path.exists() and path.stat().st_size > 0:
        print(f"Using existing {name} file: {path}")
        return path
    print(f"Downloading {name}: {spec['url']}")
    with requests.get(spec["url"], stream=True, timeout=120) as response:
        response.raise_for_status()
        with path.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return path

source_paths = {name: download_if_missing(name, spec) for name, spec in SOURCE_FILES.items()}
source_paths

Using existing collision file: C:\Users\maxis\iCloudDrive\Master\Data Stewardship\Übung\part 3\dast-2026-crash-severity-prediction\data\raw\dft-road-casualty-statistics-collision-2023.csv
Using existing vehicle file: C:\Users\maxis\iCloudDrive\Master\Data Stewardship\Übung\part 3\dast-2026-crash-severity-prediction\data\raw\dft-road-casualty-statistics-vehicle-2023.csv
Using existing casualty file: C:\Users\maxis\iCloudDrive\Master\Data Stewardship\Übung\part 3\dast-2026-crash-severity-prediction\data\raw\dft-road-casualty-statistics-casualty-2023.csv


{'collision': WindowsPath('C:/Users/maxis/iCloudDrive/Master/Data Stewardship/Übung/part 3/dast-2026-crash-severity-prediction/data/raw/dft-road-casualty-statistics-collision-2023.csv'),
 'vehicle': WindowsPath('C:/Users/maxis/iCloudDrive/Master/Data Stewardship/Übung/part 3/dast-2026-crash-severity-prediction/data/raw/dft-road-casualty-statistics-vehicle-2023.csv'),
 'casualty': WindowsPath('C:/Users/maxis/iCloudDrive/Master/Data Stewardship/Übung/part 3/dast-2026-crash-severity-prediction/data/raw/dft-road-casualty-statistics-casualty-2023.csv')}

## Transform Source Data to DBRepo Schema

In [67]:
COLLISION_COLUMNS = [
    "collision_index", "collision_year", "collision_ref_no", "location_easting_osgr",
    "location_northing_osgr", "longitude", "latitude", "police_force",
    "collision_severity", "number_of_vehicles", "number_of_casualties", "date",
    "day_of_week", "time", "local_authority_district", "local_authority_ons_district",
    "local_authority_highway", "local_authority_highway_current", "first_road_class",
    "first_road_number", "road_type", "speed_limit", "junction_detail", "junction_control",
    "second_road_class", "second_road_number", "pedestrian_crossing", "light_conditions",
    "weather_conditions", "road_surface_conditions", "special_conditions_at_site",
    "carriageway_hazards", "urban_or_rural_area", "did_police_officer_attend_scene_of_accident",
    "trunk_road_flag", "lsoa_of_accident_location", "enhanced_severity_collision",
    "collision_injury_based", "collision_adjusted_severity_serious", "collision_adjusted_severity_slight",
]

VEHICLE_COLUMNS = [
    "vehicle_id", "collision_index", "vehicle_reference", "vehicle_type",
    "towing_and_articulation", "vehicle_manoeuvre", "vehicle_direction_from", "vehicle_direction_to",
    "vehicle_location_restricted_lane", "junction_location", "skidding_and_overturning",
    "hit_object_in_carriageway", "vehicle_leaving_carriageway", "hit_object_off_carriageway",
    "first_point_of_impact", "vehicle_left_hand_drive", "journey_purpose_of_driver", "sex_of_driver",
    "age_of_driver", "age_band_of_driver", "engine_capacity_cc", "propulsion_code", "age_of_vehicle",
    "generic_make_model", "driver_imd_decile", "lsoa_of_driver", "escooter_flag", "driver_distance_banding",
]

CASUALTY_COLUMNS = [
    "casualty_id", "collision_index", "vehicle_reference", "casualty_reference", "casualty_class",
    "sex_of_casualty", "age_of_casualty", "age_band_of_casualty", "casualty_severity",
    "pedestrian_location", "pedestrian_movement", "car_passenger", "bus_or_coach_passenger",
    "pedestrian_road_maintenance_worker", "casualty_type", "casualty_imd_decile", "lsoa_of_casualty",
    "enhanced_casualty_severity", "casualty_injury_based", "casualty_adjusted_severity_serious",
    "casualty_adjusted_severity_slight", "casualty_distance_banding",
]

INTEGER_COLUMNS = {
    "collision_year", "location_easting_osgr", "location_northing_osgr", "number_of_vehicles",
    "number_of_casualties", "first_road_number", "speed_limit", "second_road_number",
    "vehicle_id", "vehicle_reference", "age_of_driver", "engine_capacity_cc", "age_of_vehicle",
    "driver_imd_decile", "casualty_id", "casualty_reference", "age_of_casualty", "casualty_imd_decile",
}
FLOAT_COLUMNS = {"longitude", "latitude"}
FLAG_COLUMNS = {
    "did_police_officer_attend_scene_of_accident", "trunk_road_flag", "collision_injury_based",
    "collision_adjusted_severity_serious", "collision_adjusted_severity_slight", "vehicle_left_hand_drive",
    "escooter_flag", "casualty_injury_based", "casualty_adjusted_severity_serious",
    "casualty_adjusted_severity_slight",
}

RENAME_ALIASES = {
    "accident_index": "collision_index",
    "accident_year": "collision_year",
    "accident_reference": "collision_ref_no",
    "collision_reference": "collision_ref_no",
    "accident_ref_no": "collision_ref_no",
    "accident_severity": "collision_severity",
    "enhanced_accident_severity": "enhanced_severity_collision",
    "enhanced_collision_severity": "enhanced_severity_collision",
    "legacy_accident_severity": "collision_severity",
    "did_police_officer_attend_scene_of_accident": "did_police_officer_attend_scene_of_accident",
    "did_police_officer_attend_scene_of_collision": "did_police_officer_attend_scene_of_accident",
    "lsoa_of_accident_location": "lsoa_of_accident_location",
    "lsoa_of_collision_location": "lsoa_of_accident_location",
}

SEVERITY_LOOKUP = {"1": "Fatal", "2": "Serious", "3": "Slight", 1: "Fatal", 2: "Serious", 3: "Slight"}

def read_source_csv(path):
    return pd.read_csv(path, dtype=str, low_memory=False).rename(columns=RENAME_ALIASES)

def ensure_columns(df, columns):
    out = df.copy()
    for column in columns:
        if column not in out.columns:
            out[column] = pd.NA
    return out[columns]

def decode_labels(df):
    out = df.copy()
    for column in ["collision_severity", "casualty_severity", "enhanced_severity_collision", "enhanced_casualty_severity"]:
        if column in out.columns:
            out[column] = out[column].map(SEVERITY_LOOKUP).fillna(out[column])
    return out

def coerce_types(df):
    out = df.copy()
    for column in out.columns.intersection(INTEGER_COLUMNS):
        out[column] = pd.to_numeric(out[column], errors="coerce").fillna(-1).astype("Int64")
    for column in out.columns.intersection(FLOAT_COLUMNS):
        out[column] = pd.to_numeric(out[column], errors="coerce").fillna(-1)
    for column in out.columns.intersection(FLAG_COLUMNS):
        normalized = out[column].astype("string").str.strip().str.lower()
        out[column] = normalized.map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
        out[column] = out[column].fillna(-1).astype("Int64")
    string_columns = out.columns.difference(list(INTEGER_COLUMNS | FLOAT_COLUMNS | FLAG_COLUMNS))
    for column in string_columns:
        out[column] = out[column].fillna("-1")
    return out

def prepare_table(name, path):
    raw = read_source_csv(path)
    if name == "collision":
        df = ensure_columns(raw, COLLISION_COLUMNS)
    elif name == "vehicle":
        if "vehicle_id" not in raw.columns:
            raw.insert(0, "vehicle_id", range(1, len(raw) + 1))
        df = ensure_columns(raw, VEHICLE_COLUMNS)
    elif name == "casualty":
        if "casualty_id" not in raw.columns:
            raw.insert(0, "casualty_id", range(1, len(raw) + 1))
        df = ensure_columns(raw, CASUALTY_COLUMNS)
    else:
        raise ValueError(name)
    df = decode_labels(df)
    df = coerce_types(df)
    return df

prepared = {name: prepare_table(name, path) for name, path in source_paths.items()}
if MAX_ROWS_PER_TABLE:
    prepared = {name: df.head(MAX_ROWS_PER_TABLE).copy() for name, df in prepared.items()}

for name, df in prepared.items():
    print(name, df.shape)
    display(df.head(3))

collision (104258, 40)


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,special_conditions_at_site,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2023170L30453,2023,170L30453,456584,522423,-1.125794,54.593835,17,Slight,3,...,4,13,2,-1,-1,E01032560,-1,0,-1,-1
1,2023111293075,2023,111293075,434218,514041,-1.472891,54.520515,11,Slight,1,...,0,0,2,1,-1,E01012346,Slight,1,0,1
2,2023111312748,2023,111312748,430616,514267,-1.528511,54.522776,11,Serious,2,...,0,0,1,-1,-1,E01012319,7,1,1,0


vehicle (189815, 28)


,vehicle_id,collision_index,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane,junction_location,...,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding
0,1,2023481356437,1,9,0,9,1,7,0,8,...,26,6,5935,1,9,-1,-1,-1,0,-1
1,2,2023440059930,1,9,0,19,6,2,0,1,...,35,6,5340,1,23,-1,7,E01027771,0,-1
2,3,2023401356150,1,9,0,11,4,7,0,8,...,-1,-1,5340,1,23,-1,-1,-1,0,-1


casualty (132977, 22)


,casualty_id,collision_index,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,...,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding
0,1,2023201345434,1,1,3,1,72,10,Slight,5,...,0,0,0,1,E01009409,Slight,1,0,1,-1
1,2,2023522400767,2,1,1,1,18,4,Slight,0,...,0,0,1,1,E01014486,-1,0,-1,-1,-1
2,3,2023010425559,1,1,1,1,21,5,Slight,0,...,0,0,3,3,E01001391,-1,0,0,1,-1


## Key Integrity and 3NF Checks

In [68]:
def assert_unique(df, columns, label):
    duplicate_count = int(df.duplicated(columns, keep=False).sum())
    if duplicate_count:
        raise AssertionError(f"{label} is not unique: {duplicate_count} duplicated rows")
    return {"check": f"{label} unique", "status": "pass", "detail": "0 duplicated rows"}

def assert_fk(child, parent, child_cols, parent_cols, label):
    child_keys = child[list(child_cols)].dropna().drop_duplicates()
    parent_keys = parent[list(parent_cols)].dropna().drop_duplicates()
    merged = child_keys.merge(parent_keys, left_on=list(child_cols), right_on=list(parent_cols), how="left", indicator=True)
    missing = merged[merged["_merge"] == "left_only"]
    if len(missing):
        raise AssertionError(f"{label} has {len(missing)} missing parent keys")
    return {"check": label, "status": "pass", "detail": "all child keys found"}

validation_rows = []
validation_rows.append(assert_unique(prepared["collision"], ["collision_index"], "collision.collision_index"))
validation_rows.append(assert_unique(prepared["vehicle"], ["collision_index", "vehicle_reference"], "vehicle natural key"))
validation_rows.append(assert_unique(prepared["casualty"], ["collision_index", "casualty_reference"], "casualty natural key"))
validation_rows.append(assert_fk(prepared["vehicle"], prepared["collision"], ["collision_index"], ["collision_index"], "vehicle.collision_index -> collision.collision_index"))
validation_rows.append(assert_fk(prepared["casualty"], prepared["collision"], ["collision_index"], ["collision_index"], "casualty.collision_index -> collision.collision_index"))
validation_rows.append(assert_fk(prepared["casualty"], prepared["vehicle"], ["collision_index", "vehicle_reference"], ["collision_index", "vehicle_reference"], "casualty vehicle_reference -> vehicle natural key"))

schema_validation = pd.DataFrame(validation_rows)
display(schema_validation)

,check,status,detail
0,collision.collision_index unique,pass,0 duplicated rows
1,vehicle natural key unique,pass,0 duplicated rows
2,casualty natural key unique,pass,0 duplicated rows
3,vehicle.collision_index -> collision.collision...,pass,all child keys found
4,casualty.collision_index -> collision.collisio...,pass,all child keys found
5,casualty vehicle_reference -> vehicle natural key,pass,all child keys found


Short 3NF assessment for the report:

| Table | Grain | Primary key | 3NF assessment |
|---|---|---|---|
| `collision` | One row per collision | `collision_index` | Collision attributes depend on the collision key; vehicle and casualty details are separated. |
| `vehicle` | One row per vehicle in a collision | `vehicle_id`; natural key `collision_index + vehicle_reference` | Vehicle attributes depend on the vehicle instance, and collision facts stay in `collision`. |
| `casualty` | One row per casualty | `casualty_id`; natural key `collision_index + casualty_reference` | Casualty attributes depend on the casualty instance, and vehicle linkage is represented by `collision_index + vehicle_reference`. |

## DBRepo Table Lookup and Duplicate-Load Check

In [69]:
def request_json(method, path, **kwargs):
    response = requests.request(method, f"{API_BASE}{path}", auth=AUTH, timeout=120, **kwargs)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed: {response.status_code} {response.text}")
    if not response.text:
        return None
    return response.json()

def table_row_count_or_probe(table_id):
    response = requests.head(f"{API_BASE}/database/{DBREPO_DB_ID}/table/{table_id}/data", auth=AUTH, timeout=60)
    if response.status_code < 400 and response.headers.get("X-Count") is not None:
        return int(response.headers["X-Count"])
    data = requests.get(f"{API_BASE}/database/{DBREPO_DB_ID}/table/{table_id}/data", auth=AUTH, params={"page": 0, "size": 1}, timeout=60)
    if data.status_code >= 400:
        raise RuntimeError(f"Could not count table {table_id}: {data.status_code} {data.text}")
    rows = data.json()
    if isinstance(rows, dict) and "data" in rows:
        rows = rows["data"]
    return len(rows)

tables = request_json("GET", f"/database/{DBREPO_DB_ID}/table")
table_lookup = {table["name"]: table for table in tables if table.get("name") in prepared}

missing_tables = set(prepared) - set(table_lookup)
if missing_tables:
    raise RuntimeError(f"Missing DBRepo target tables: {sorted(missing_tables)}")

existing_counts = {name: table_row_count_or_probe(table["id"]) for name, table in table_lookup.items()}
display(pd.DataFrame([{"table": k, "existing_rows_or_probe": v} for k, v in existing_counts.items()]))

if any(count > 0 for count in existing_counts.values()) and LOAD_TO_DBREPO and not ALLOW_NONEMPTY_LOAD:
    raise RuntimeError("At least one target table already contains rows. Set ALLOW_NONEMPTY_LOAD=true only after an explicit cleanup/reload decision.")

,table,existing_rows_or_probe
0,vehicle,189815
1,collision,104258
2,casualty,132977


## Load Data into DBRepo

Set `LOAD_TO_DBREPO=true` to execute the upload. By default this cell performs a dry run so the verification code can be reviewed safely before loading.

In [70]:
def import_dataframe(table_name, df):
    table_id = table_lookup[table_name]["id"]
    import_df = df.copy()
    import_df = import_df.where(pd.notna(import_df), None)
    CLIENT.import_table_data(database_id=DBREPO_DB_ID, table_id=table_id, dataframe=import_df)
    return len(import_df)

load_rows = []
for table_name in ["collision", "vehicle", "casualty"]:
    expected_rows = len(prepared[table_name])
    if LOAD_TO_DBREPO:
        inserted_rows = import_dataframe(table_name, prepared[table_name])
        status = "imported_with_dbrepo_client"
    else:
        inserted_rows = 0
        status = "dry_run"
    load_rows.append({"table": table_name, "expected_rows": expected_rows, "inserted_rows": inserted_rows, "status": status})

load_report = pd.DataFrame(load_rows)
display(load_report)

,table,expected_rows,inserted_rows,status
0,collision,104258,0,dry_run
1,vehicle,189815,0,dry_run
2,casualty,132977,0,dry_run


## Local Expected Results for T2.4 Views

In [71]:
def local_v_ml_features(tables):
    cas = tables["casualty"]
    c = tables["collision"]
    v = tables["vehicle"]
    merged = cas.merge(c, on="collision_index", suffixes=("", "_collision"))
    merged = merged.merge(v[["collision_index", "vehicle_reference", "vehicle_type"]], on=["collision_index", "vehicle_reference"])
    return merged[[
        "casualty_id", "collision_index", "casualty_severity", "road_type", "speed_limit",
        "weather_conditions", "light_conditions", "road_surface_conditions", "time", "day_of_week",
        "number_of_vehicles", "vehicle_type",
    ]]

def local_v_severity_distribution(tables):
    counts = tables["casualty"].groupby("casualty_severity", dropna=False).size().reset_index(name="total_count")
    counts["percentage"] = (counts["total_count"] * 100.0 / counts["total_count"].sum()).round(2)
    return counts.sort_values("total_count", ascending=False).reset_index(drop=True)

def worst_severity(values):
    values = set(values.dropna())
    if "Fatal" in values:
        return "Fatal"
    if "Serious" in values:
        return "Serious"
    return "Slight"

def local_v_collision_summary(tables):
    c = tables["collision"]
    cas = tables["casualty"]
    base = c.merge(cas[["collision_index", "casualty_severity"]], on="collision_index")
    group_cols = ["collision_index", "date", "day_of_week", "time", "road_type", "speed_limit", "weather_conditions", "light_conditions", "road_surface_conditions", "number_of_vehicles", "number_of_casualties"]
    return base.groupby(group_cols, dropna=False)["casualty_severity"].apply(worst_severity).reset_index(name="worst_severity")

def local_v_feature_null_check(tables):
    checks = [
        ("collision", "road_type"), ("collision", "speed_limit"), ("collision", "weather_conditions"),
        ("collision", "light_conditions"), ("collision", "road_surface_conditions"), ("collision", "time"),
        ("collision", "day_of_week"), ("collision", "number_of_vehicles"), ("vehicle", "vehicle_type"),
        ("casualty", "casualty_severity"),
    ]
    rows = []
    for table_name, column in checks:
        series = tables[table_name][column]
        rows.append({"feature": column, "null_count": int(series.isna().sum()), "total": int(len(series))})
    return pd.DataFrame(rows)

expected_views = {
    "v_ml_features": local_v_ml_features(prepared),
    "v_severity_distribution": local_v_severity_distribution(prepared),
    "v_collision_summary": local_v_collision_summary(prepared),
    "v_feature_null_check": local_v_feature_null_check(prepared),
}

for name, df in expected_views.items():
    print(name, df.shape)
    display(df.head(3))

v_ml_features (132977, 12)


,casualty_id,collision_index,casualty_severity,road_type,speed_limit,weather_conditions,light_conditions,road_surface_conditions,time,day_of_week,number_of_vehicles,vehicle_type
0,1,2023201345434,Slight,6,30,2,1,2,20:08,7,1,9
1,2,2023522400767,Slight,6,20,9,1,1,10:38,3,2,1
2,3,2023010425559,Slight,6,20,1,4,1,00:30,1,3,3


v_severity_distribution (3, 3)


,casualty_severity,total_count,percentage
0,Slight,105323,79.20
1,Serious,26030,19.57
2,Fatal,1624,1.22


v_collision_summary (104258, 12)


,collision_index,date,day_of_week,time,road_type,speed_limit,weather_conditions,light_conditions,road_surface_conditions,number_of_vehicles,number_of_casualties,worst_severity
0,2023010419171,01/01/2023,1,01:24,2,20,8,4,2,1,1,Slight
1,2023010419183,01/01/2023,1,02:25,6,30,1,4,1,3,2,Slight
2,2023010419189,01/01/2023,1,03:50,1,30,1,4,1,2,1,Slight


v_feature_null_check (10, 3)


,feature,null_count,total
0,road_type,0,104258
1,speed_limit,0,104258
2,weather_conditions,0,104258


## Compare DBRepo Views with Local Expected Results

In [72]:
def get_views():
    views = request_json("GET", f"/database/{DBREPO_DB_ID}/view")
    return {view["name"]: view for view in views}

def get_view_data(view_id, page_size=10000):
    all_rows = []
    page = 0
    while True:
        response = requests.get(
            f"{API_BASE}/database/{DBREPO_DB_ID}/view/{view_id}/data",
            auth=AUTH,
            params={"page": page, "size": page_size},
            headers={"Accept": "application/json"},
            timeout=120,
        )
        if response.status_code >= 400:
            raise RuntimeError(f"View data request failed for {view_id}: {response.status_code} {response.text}")
        rows = response.json()
        if isinstance(rows, dict) and "data" in rows:
            rows = rows["data"]
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < page_size:
            break
        page += 1
    return pd.DataFrame(all_rows)

def normalize_for_compare(df):
    out = df.copy()
    for column in out.columns:
        out[column] = out[column].astype("string").fillna("<NA>")
    return out

views = get_views()
missing_views = set(expected_views) - set(views)
if missing_views:
    view_verification = pd.DataFrame([
        {
            "view": view_name,
            "expected_rows": len(expected_views[view_name]),
            "actual_rows": pd.NA,
            "same_columns": False,
            "same_row_count": False,
            "sample_rows_match": False,
            "status": "pending: DBRepo view not found",
        }
        for view_name in sorted(missing_views)
    ])
    display(view_verification)
    print("Local expected view results were computed, but these T2.4 views are not present in DBRepo yet.")
    print("Create/register the T2.4 views in DBRepo before final T2.5 verification.")
else:
    view_checks = []
    for view_name, expected_df in expected_views.items():
        actual_df = get_view_data(views[view_name]["id"])
        expected_columns = list(expected_df.columns)
        actual_columns = list(actual_df.columns)
        same_columns = expected_columns == actual_columns
        same_row_count = len(expected_df) == len(actual_df)

        sample_match = False
        if same_columns and len(actual_df) and len(expected_df):
            expected_sample = normalize_for_compare(expected_df[expected_columns].head(20)).reset_index(drop=True)
            actual_sample = normalize_for_compare(actual_df[expected_columns].head(20)).reset_index(drop=True)
            sample_match = expected_sample.equals(actual_sample)

        view_checks.append({
            "view": view_name,
            "expected_rows": len(expected_df),
            "actual_rows": len(actual_df),
            "same_columns": same_columns,
            "same_row_count": same_row_count,
            "sample_rows_match": sample_match,
            "status": "checked",
        })

    view_verification = pd.DataFrame(view_checks)
    display(view_verification)

    if not view_verification[["same_columns", "same_row_count"]].all().all():
        raise AssertionError("At least one DBRepo view does not match the expected local columns or row count.")


,view,expected_rows,actual_rows,same_columns,same_row_count,sample_rows_match,status
0,v_collision_summary,104258,<NA>,False,False,False,pending: DBRepo view not found
1,v_feature_null_check,10,<NA>,False,False,False,pending: DBRepo view not found
2,v_ml_features,132977,<NA>,False,False,False,pending: DBRepo view not found
3,v_severity_distribution,3,<NA>,False,False,False,pending: DBRepo view not found


Local expected view results were computed, but these T2.4 views are not present in DBRepo yet.
Create/register the T2.4 views in DBRepo before final T2.5 verification.
